005 DATA CLEANING

- Start with 004_data_only_pedestrians
- End with 005_data_only_pedestrians

OBJECTIVES: 
- Resolve any issues with 'TipoPersona' values
- Transfer driver & passenger information to the pedestrian rows
- Delete the driver & passenger rows
- Final product: only pedestrian rows containing vehicle, driver and passenger information

In [194]:
import pandas as pd
import numpy as np
import pyarrow

In [195]:
df = pd.read_parquet("004_data_only_pedestrians.parquet").copy()
df.shape

(40982, 35)

The dataset currently has 35 columns and 40982 rows of data. 
Examining the 'TipoPersona' column:

In [196]:
df['TipoPersona'].value_counts()

TipoPersona
Pedone                   19713
Conducente               17633
Passeggero                3630
Pedone sconosciuto           4
Passeggero Istruttore        2
Name: count, dtype: int64

In [197]:
df.columns

Index(['Protocollo', 'Progressivo', 'TipoPersona', 'TipoVeicolo',
       'StatoVeicolo', 'NaturaIncidente', 'NUM_FERITI', 'NUM_RISERVATA',
       'NUM_MORTI', 'NUM_ILLESI', 'Sesso', 'DataOraIncidente',
       'CondizioneAtmosferica', 'Traffico', 'Gruppo', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'Latitude', 'Longitude', 'particolaritastrade',
       'TipoStrada', 'FondoStradale', 'Pavimentazione', 'Segnaletica',
       'Visibilita', 'Illuminazione', 'Tipolesione', 'Deceduto',
       'DecedutoDopo', 'parked_vehicle_involved'],
      dtype='object')

DATA CLEANING - FIND ACCIDENTS WITH NO DRIVER

In [198]:
# Find protocols that have any driver
protocols_with_driver = df[df['TipoPersona']
                           == 'Conducente']['Protocollo'].unique()

# Now find protocols that do not have any driver
all_protocols = df['Protocollo'].unique()

protocols_no_driver = set(all_protocols) - set(protocols_with_driver)
protocols_no_driver = list(protocols_no_driver)

print(f"Number of protocols with no driver: {len(protocols_no_driver)}")

Number of protocols with no driver: 600


In [199]:
df.loc[df['Protocollo'].isin(protocols_no_driver), df.columns[:6]].head()

,Protocollo,Progressivo,TipoPersona,TipoVeicolo,StatoVeicolo,NaturaIncidente
5,1931673,<NA>,Pedone,,,Investimento di pedone
6,1931673,<NA>,Pedone,,,Investimento di pedone
31,1931793,<NA>,Pedone,,,Investimento di pedone
111,1932278,<NA>,Pedone,,,Investimento di pedone
142,1932512,<NA>,Pedone,,,Investimento di pedone


Although we are missing the driver, driver sex, vehicle and driver injury (and potentially passenger) data, we do have the time of accident for these protocols. We shall endeavour to include these accidents. 

For these accidents, we will set:
- TipoVeicolo == NA
- StatoVeicolo == Allontanatosi
- driver_unknown_sex == 1

In [200]:
# Create a mask for protocols_no_driver
mask_protocols_no_driver = df['Protocollo'].isin(protocols_no_driver)

# Apply the changes
df.loc[mask_protocols_no_driver, 'TipoVeicolo'] = 'Unknown'
df.loc[mask_protocols_no_driver, 'StatoVeicolo'] = 'Allontanatosi'

# If column doesn't exist, create it
if 'driver_unknown_sex' not in df.columns:
    df['driver_unknown_sex'] = 0

df.loc[mask_protocols_no_driver, 'driver_unknown_sex'] = 1

print(f"Updated {mask_protocols_no_driver.sum()} rows in protocols_no_driver.")

Updated 641 rows in protocols_no_driver.


Check:

In [201]:
cols_to_show = [
    "Protocollo",
    "Progressivo",
    "TipoPersona",
    "TipoVeicolo",
    "StatoVeicolo",
    "driver_unknown_sex"
]

df.loc[df["Protocollo"].isin(protocols_no_driver), cols_to_show].head()

,Protocollo,Progressivo,TipoPersona,TipoVeicolo,StatoVeicolo,driver_unknown_sex
5,1931673,<NA>,Pedone,Unknown,Allontanatosi,1
6,1931673,<NA>,Pedone,Unknown,Allontanatosi,1
31,1931793,<NA>,Pedone,Unknown,Allontanatosi,1
111,1932278,<NA>,Pedone,Unknown,Allontanatosi,1
142,1932512,<NA>,Pedone,Unknown,Allontanatosi,1


DATA CLEANING - COPY DRIVER SEX, VEHICLE AND STATO VEHICLE TO PEDESTRIANS WITHIN EACH ACCIDENT PROTOCOL

First we double-check for accidents with more than 1 driver:

In [202]:
# Count how many drivers ("Conducente") per protocol
drivers_per_protocol = (
    df.loc[df["TipoPersona"] == "Conducente"]
      .groupby("Protocollo")["TipoPersona"]
      .count()
)

# Keep only protocols with >1 driver
multiple_drivers = drivers_per_protocol[drivers_per_protocol > 1]

print(f"Protocols with multiple drivers: {len(multiple_drivers)}")

if not multiple_drivers.empty:
    print("Examples:")
    print(multiple_drivers.head())

Protocols with multiple drivers: 0


In [203]:
df[["Protocollo", "Progressivo", "TipoPersona", "TipoVeicolo", "StatoVeicolo"]].head()

,Protocollo,Progressivo,TipoPersona,TipoVeicolo,StatoVeicolo
0,1931584,<NA>,Pedone,,
1,1931584,1,Conducente,Autocarro inferiore 35 q.,In marcia / fermata / arresto
2,1931584,1,Passeggero,Autocarro inferiore 35 q.,In marcia / fermata / arresto
3,1931666,<NA>,Pedone,,
4,1931666,1,Conducente,Autovettura pubblica,In marcia / fermata / arresto


Now we will copy the TipoVeicolo, StatoVeicolo from the 'Conducente' row onto the 'Pedone' row(s) for each protocol. 

We will also copy the driver Tipolesione, driver Sesso, driver Deceduto and driver DecedutoDopo into a new dedicated column (on pedestrian rows) for each protocol. 

In [204]:
# Find driver information
driver_lookup = (
    df.loc[df["TipoPersona"] == "Conducente",
           ["Protocollo", "TipoVeicolo", "StatoVeicolo", "Tipolesione", "Deceduto", "DecedutoDopo", "Sesso"]]
    .drop_duplicates("Protocollo")
    .set_index("Protocollo")
    .add_prefix("driver_")  # -> driver_TipoVeicolo, driver_StatoVeicolo, etc
)

# Merge using the protocols so new columns are added to the df: driver_StatoVeicolo, etc
df = df.merge(driver_lookup, left_on="Protocollo",
              right_index=True, how="left")

# Copy driver vehicle/state ONLY to pedestrians in protocols that have a driver
is_ped = df["TipoPersona"].eq("Pedone")
has_driver = df["driver_TipoVeicolo"].notna(
) | df["driver_StatoVeicolo"].notna()

df.loc[is_ped & has_driver, "TipoVeicolo"] = df.loc[is_ped &
                                                    has_driver, "driver_TipoVeicolo"]
df.loc[is_ped & has_driver, "StatoVeicolo"] = df.loc[is_ped &
                                                     has_driver, "driver_StatoVeicolo"]

# Rename merged injury/death columns to your target names
df = df.rename(columns={
    "driver_Tipolesione":   "driver_injury",
    "driver_Deceduto":      "driver_deceduto",
    "driver_DecedutoDopo":  "driver_deceduto_dopo"
})

# Normalize driver sex into M, F, U (unknown)
df["driver_Sesso"] = df["driver_Sesso"].map(
    {"M": "M", "F": "F"}).fillna("U").replace("", "U")

# Checks:
protocols_with_drivers = df["driver_injury"].notna().groupby(
    df["Protocollo"]).any().sum()
pedestrian_updates = (is_ped & has_driver).sum()
rows_with_driver_info = df["driver_injury"].notna().sum()

print(f"Pedestrian vehicle/state updates: {pedestrian_updates}")
print(f"Rows with driver info merged:    {rows_with_driver_info}")
print(f"Protocols with drivers:          {protocols_with_drivers}")
print("Driver info successfully merged")

Pedestrian vehicle/state updates: 19077
Rows with driver info merged:    40341
Protocols with drivers:          17633
Driver info successfully merged


In [205]:
df = df.drop(columns=["driver_TipoVeicolo", "driver_StatoVeicolo"])

To avoid ending up with '' as well as NAN, we replace '' with NAN.

In [206]:
df['driver_deceduto_dopo'] = df['driver_deceduto_dopo'].replace('', pd.NA)
df['driver_injury'] = df['driver_injury'].replace('', pd.NA)

Now we need to make driver_Sesso = U where driver_unknown_sex = 1

In [207]:
df.loc[df["driver_unknown_sex"] == 1, "driver_Sesso"] = "U"
df = df.drop(columns="driver_unknown_sex")

DATA CLEANING: PASSENGERS

I would like to retain information about the passengers:
- sex - how many of each
- injured or uninjured or killed - how many of each

In [208]:
# Define passenger types and find passenger data
passenger_types = ["Passeggero", "Passeggero Istruttore",
                   "Passeggero non identificato"]
passenger_data = df.loc[df["TipoPersona"].isin(passenger_types),
                        ["Protocollo", "Sesso", "Tipolesione"]].copy()

# Initialize columns with passenger information
target_cols = [
    "passenger_no_of_females", "passenger_no_of_males", "passenger_no_of_unknown_sex",
    "passengers_killed", "passengers_uninjured", "passengers_injured"
]
for c in target_cols:
    if c not in df.columns:
        df[c] = 0

if passenger_data.empty:
    print("Protocols with passengers: 0")
    print("Passenger info updates (all rows): 0")
    print("All passenger information successfully processed.")
else:
    # Categorize sex & injury
    passenger_data["sex_category"] = passenger_data["Sesso"].map(
        {"F": "female", "M": "male"}).fillna("unknown")

    injury_map = {
        "Deceduto durante prime cure": "killed",
        "Deceduto durante trasporto":  "killed",
        "Deceduto sul posto":          "killed",
        "Illeso":                      "uninjured",
        "Rifiuta cure immediate":      "uninjured",
        "Prognosi riservata":          "injured",
        "Ricoverato":                  "injured",
        "Rimandato":                   "injured",
    }
    passenger_data["injury_category"] = passenger_data["Tipolesione"].map(
        injury_map)

    # Counts per protocol via pivot_table ---
    sex_counts = (
        passenger_data.pivot_table(index="Protocollo",
                                   columns="sex_category",
                                   values="Sesso",
                                   aggfunc="size",
                                   fill_value=0)
        .reindex(columns=["female", "male", "unknown"], fill_value=0)
        .rename(columns={
            "female": "passenger_no_of_females",
            "male": "passenger_no_of_males",
            "unknown": "passenger_no_of_unknown_sex"
        })
    )

    injury_counts = (
        passenger_data.pivot_table(index="Protocollo",
                                   columns="injury_category",
                                   values="Tipolesione",
                                   aggfunc="size",
                                   fill_value=0)
        .reindex(columns=["killed", "uninjured", "injured"], fill_value=0)
        .rename(columns={
            "killed": "passengers_killed",
            "uninjured": "passengers_uninjured",
            "injured": "passengers_injured"
        })
    )

    per_protocol_counts = (
        sex_counts.join(injury_counts, how="outer").fillna(0).astype("int16")
    )

    # Merge to ALL rows of those protocols
    df = df.drop(columns=[c for c in target_cols if c in df.columns]).merge(
        per_protocol_counts, left_on="Protocollo", right_index=True, how="left"
    )

    # Fill protocols without passengers with zeros and ensure compact dtype
    df[target_cols] = df[target_cols].fillna(0).astype("int16")

    # Checks
    prot_with_passengers = per_protocol_counts.shape[0]
    rows_updated = df["Protocollo"].isin(per_protocol_counts.index).sum()

    print(f"Protocols with passengers: {prot_with_passengers}")
    print(f"Passenger info updates (all rows): {rows_updated}")
    print(f"\nTotal passengers processed: {len(passenger_data)}")
    print("\nSex distribution:")
    print(passenger_data["sex_category"].value_counts())
    print("\nInjury distribution:")
    print(passenger_data["injury_category"].value_counts(dropna=False))
    print("\nAll passenger information successfully processed.")

Protocols with passengers: 2910
Passenger info updates (all rows): 9705

Total passengers processed: 3632

Sex distribution:
sex_category
female     2061
male       1565
unknown       6
Name: count, dtype: int64

Injury distribution:
injury_category
uninjured    3517
injured       115
Name: count, dtype: int64

All passenger information successfully processed.


In [209]:
df.columns

Index(['Protocollo', 'Progressivo', 'TipoPersona', 'TipoVeicolo',
       'StatoVeicolo', 'NaturaIncidente', 'NUM_FERITI', 'NUM_RISERVATA',
       'NUM_MORTI', 'NUM_ILLESI', 'Sesso', 'DataOraIncidente',
       'CondizioneAtmosferica', 'Traffico', 'Gruppo', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'Latitude', 'Longitude', 'particolaritastrade',
       'TipoStrada', 'FondoStradale', 'Pavimentazione', 'Segnaletica',
       'Visibilita', 'Illuminazione', 'Tipolesione', 'Deceduto',
       'DecedutoDopo', 'parked_vehicle_involved', 'driver_injury',
       'driver_deceduto', 'driver_deceduto_dopo', 'driver_Sesso',
       'passenger_no_of_females', 'passenger_no_of_males',
       'passenger_no_of_unknown_sex', 'passengers_killed',
       'passengers_uninjured', 'passengers_injured'],
      dtype='object')

In [210]:
cols_to_show = [
    "Protocollo",
    "Progressivo",
    "TipoPersona",
    "Sesso",
    "Tipolesione"
]

protocols_pedestrian_unknown = df[df['TipoPersona']
                                  == 'Pedone sconosciuto']['Protocollo']

df.loc[df['Protocollo'].isin(protocols_pedestrian_unknown), cols_to_show]

,Protocollo,Progressivo,TipoPersona,Sesso,Tipolesione
20369,3463571,<NA>,Pedone,M,Rimandato
20370,3463571,<NA>,Pedone sconosciuto,,Illeso
20371,3463571,1,Conducente,M,Illeso
33143,5114036,<NA>,Pedone,M,Rimandato
33144,5114036,<NA>,Pedone sconosciuto,,Illeso
33145,5114036,1,Conducente,M,Illeso
36497,5605020,<NA>,Pedone,M,Rimandato
36498,5605020,<NA>,Pedone sconosciuto,,Illeso
36499,5605020,1,Conducente,F,Illeso
40326,6009160,<NA>,Pedone,M,Rimandato


Where there is a 'Pedone sconosciuto', a pedestrian is involved, but we have no follow up information about their injuries. We will delete these unknown pedestrians, preserving the information about the other pedestrians involved in each accident.

Now we have transferred information about the driver, passengers, vehicle and state of vehicle to the pedestrian rows, we can also delete the rows including drivers and passengers. 

In [211]:
# Define rows to delete
rows_to_delete = ['Conducente', 'Passeggero', 'Passeggero Istruttore',
                  'Passeggero non identificato', 'Pedone sconosciuto']

# Count rows before deletion
rows_before = len(df)
rows_to_delete_count = df['TipoPersona'].isin(rows_to_delete).sum()

# Delete the rows
df = df[~df['TipoPersona'].isin(rows_to_delete)].reset_index(drop=True)

# Count rows after deletion
rows_after = len(df)

print(f"Rows before deletion: {rows_before}")
print(f"Rows deleted: {rows_to_delete_count}")
print(f"Rows after deletion: {rows_after}")
print(f"\nVerification: {rows_before - rows_to_delete_count == rows_after}")
print("Deletion completed successfully.")

Rows before deletion: 40982
Rows deleted: 21269
Rows after deletion: 19713

Verification: True
Deletion completed successfully.


In [212]:
df['TipoPersona'].value_counts(dropna=False)

TipoPersona
Pedone    19713
Name: count, dtype: int64

In [213]:
df['Progressivo'].value_counts(dropna=False)

Progressivo
<NA>    19713
Name: count, dtype: Int64

The TipoPersona and Progressivo columns now add no extra information. We can delete them.

In [214]:
df = df.drop(columns=['TipoPersona', 'Progressivo'], errors='ignore')

Now we have 19713 pedestrians to examine. 

In [215]:
df.columns

Index(['Protocollo', 'TipoVeicolo', 'StatoVeicolo', 'NaturaIncidente',
       'NUM_FERITI', 'NUM_RISERVATA', 'NUM_MORTI', 'NUM_ILLESI', 'Sesso',
       'DataOraIncidente', 'CondizioneAtmosferica', 'Traffico', 'Gruppo',
       'Localizzazione1', 'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02',
       'Chilometrica', 'DaSpecificare', 'Latitude', 'Longitude',
       'particolaritastrade', 'TipoStrada', 'FondoStradale', 'Pavimentazione',
       'Segnaletica', 'Visibilita', 'Illuminazione', 'Tipolesione', 'Deceduto',
       'DecedutoDopo', 'parked_vehicle_involved', 'driver_injury',
       'driver_deceduto', 'driver_deceduto_dopo', 'driver_Sesso',
       'passenger_no_of_females', 'passenger_no_of_males',
       'passenger_no_of_unknown_sex', 'passengers_killed',
       'passengers_uninjured', 'passengers_injured'],
      dtype='object')

In [216]:
df.to_parquet('005_data_only_pedestrians.parquet', index=False)

In [ ]:
metadata = {
    'notebook': '005 Data cleaning columns.ipynb',
    'step': 'column engineering (driver/passenger) + keep only pedestrians',
    'initial_parquet_file': '004_data_only_pedestrians.parquet',
    'initial_rows': 40982,
    'initial_columns': 35,

    # Columns
    'columns_added': [
        'driver_injury', 'driver_deceduto', 'driver_deceduto_dopo', 'driver_Sesso',
        'passenger_no_of_females', 'passenger_no_of_males', 'passenger_no_of_unknown_sex',
        'passengers_killed', 'passengers_uninjured', 'passengers_injured'
    ],
    'columns_removed': [
        'driver_TipoVeicolo', 'driver_StatoVeicolo', 'driver_unknown_sex',
        'TipoPersona', 'Progressivo'
    ],
    'columns_renamed': {
        'driver_Tipolesione': 'driver_injury',
        'driver_Deceduto': 'driver_deceduto',
        'driver_DecedutoDopo': 'driver_deceduto_dopo'
    },

    # Shapes along the way (rows, cols)
    'intermediate_shapes': [
        ('read', (40982, 35)),
        ('after_driver_passenger_engineering', ('rows', 40982, 'cols', 45)),
        ('after_keep_only_pedestrians', (19713, 45)),
        ('final_after_drop_cols', (19713, 43))
    ],

    # Driver merge & fixes
    'driver_merge_summary': {
        'pedestrian_vehicle_state_updates': 19077,
        'rows_with_driver_info_merged': 40341,
        'protocols_with_drivers': 17633
    },
    'protocols_no_driver_summary': {
        'protocols_no_driver': 600,
        'rows_updated': 641,
        'fields_set': {'TipoVeicolo': 'Unknown', 'StatoVeicolo': 'Allontanatosi', 'driver_Sesso': 'U'}
    },

    # Passenger aggregation (per protocol) merged back to all rows of those protocols
    'passenger_summary': {
        'protocols_with_passengers': 2910,
        'rows_updated_all_rows': 9705,
        'total_passengers_processed': 3632,
        'sex_distribution': {'female': 2061, 'male': 1565, 'unknown': 6},
        'injury_distribution': {'uninjured': 3517, 'injured': 115}
    },

    # Result
    'rows_removed': 21269,
    'rows_removed_share_%': 51.90,
    'final_rows': 19713,
    'final_columns': 43,
    'final_parquet_file': '005_data_only_pedestrians.parquet',

    'decisions_made': [
        'Merged driver info into pedestrian rows; standardized driver injury/death fields and marked unknown driver sex for protocols lacking a driver.',
        'Computed per-protocol passenger counts by sex and injury outcome; merged into all rows for those protocols.',
        'Filtered to pedestrian-only rows, effectively moving toward one-row-per-accident representation.',
        'Dropped helper/role columns (TipoPersona, Progressivo) post-filtering.',
        'Removed temporary driver helper columns after transferring values to pedestrian rows.'
    ]
}